# Поиск по HOM-литературе — подготовленная копия, без результатов
LLM/Яндекс здесь не используются. OpenAlex, Wikidata и загрузка кодировщика требуют сети.
Кодировщик работает на CPU, чтобы не конкурировать с Ollama за видеопамять.
Исходные материалы курса сохранены; пути вывода направлены в reports/ и library_search/ проекта.
Пять запросов по теме предложены ассистентом: прочитайте и при необходимости измените.
Пять комментариев [написано] заполняет пользователь после фактического просмотра результатов.
Сетевые ошибки, которые учебный ноутбук печатает и пропускает, означают незавершённый соответствующий шаг.

# Неделя 5. Поиск по своим источникам и внешние графы знаний

Ноутбук показывает на одной учебной библиотеке, как устроены три модели поиска —
TF-IDF, BM25 и векторный поиск, — как измерить их качество и как обратиться к внешним
графам знаний: Wikidata (SPARQL) и OpenAlex (API). В конце запросы оформлены как
инструменты MCP, которыми пользуется учебный агент.

**Как работать.** Ячейки выполняются сверху вниз. Писать код не нужно: достаточно
запустить ячейки, прочитать пояснения и поменять запросы там, где это предложено.

**Что нужно.** Python 3.10+, пакеты `numpy`, `scikit-learn`, `matplotlib`,
`sentence-transformers` (для последнего раздела — `mcp`). Первая загрузка кодировщика —
около 470 МБ. Сеть нужна для разделов 11–13; при её отсутствии ячейки сообщают об этом и
ноутбук выполняется дальше.

| Раздел | Содержание |
|---|---|
| 1–2 | Учебная библиотека, нарезка на фрагменты |
| 3–5 | Инвертированный индекс, TF-IDF, BM25, опыт с одним запросом и двумя моделями |
| 6–8 | Векторный поиск, проекция с кластерами, поиск через кластеры |
| 9–10 | Метрики качества, слияние выдач, переранжирование |
| 11–12 | Wikidata и SPARQL, OpenAlex |
| 13 | Запросы как инструменты MCP |
| 14 | Своя тема: библиотека из OpenAlex и пять своих запросов |

In [1]:
import json, math, os, re, time, urllib.error, urllib.parse, urllib.request, warnings
from collections import Counter, defaultdict
from datetime import date
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/quick_fit.py").is_file())
KIT = PROJECT / "ollama_kit"
import sys
sys.path.insert(0, str(KIT / "week5"))
LIB = KIT / "week5/library_demo"                        # учебная библиотека
MODEL_NAME = "intfloat/multilingual-e5-small"     # кодировщик: 118 млн параметров, вектор из 384 чисел
# Для медленной сети: "sentence-transformers/all-MiniLM-L6-v2" (около 90 МБ, только английский язык)

warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
np.random.seed(0)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})

ModuleNotFoundError: No module named 'matplotlib'

## 1. Учебная библиотека

80 научных работ по четырём темам, по 20 на тему. Записи получены из OpenAlex; запросы и
дата обращения лежат рядом, в `library_demo/corpus_provenance.json`. Единица поиска —
документ: название и аннотация.

In [2]:
docs = [json.loads(line) for line in (LIB / "corpus.jsonl").open(encoding="utf-8")]
for d in docs:
    d["text"] = d["title"] + ". " + d["abstract"]
topics = sorted({d["topic"] for d in docs})
print("документов:", len(docs))
for t in topics:
    words = [len(d["text"].split()) for d in docs if d["topic"] == t]
    print(f"  {t:24s} {len(words)} работ, слов в документе в среднем {sum(words) // len(words)}")
d = docs[0]
print("\nПример записи:", d["id"], "|", d["title"], "|", d["year"], "|", d["source"])
print(d["abstract"][:300], "…")
prov = json.loads((LIB / "corpus_provenance.json").read_text(encoding="utf-8"))
print("\nПроисхождение:", prov["source"], "| лицензия", prov["license"], "| дата обращения", prov["retrieved"])

NameError: name 'LIB' is not defined

OpenAlex хранит аннотацию **инвертированным индексом**: для каждого слова — список его
позиций в тексте. Эту же структуру данных использует лексический поиск (раздел 3).
Ячейка ниже запрашивает одну запись и восстанавливает текст аннотации.

In [3]:
OPENALEX = "https://api.openalex.org"
OPENALEX_KEY = os.environ.get("OPENALEX_API_KEY", "")     # ключ — в переменной окружения, в ноутбук его не пишем
USER_AGENT = "AI4ScienceCourse/0.1 (search_lab)"

def openalex_get(path, **params):
    # Запрос к OpenAlex. Возвращает ответ и адрес запроса без ключа (для записи о происхождении).
    public = f"{OPENALEX}{path}" + ("?" + urllib.parse.urlencode(params) if params else "")
    if OPENALEX_KEY:
        params = {**params, "api_key": OPENALEX_KEY}
    req = urllib.request.Request(f"{OPENALEX}{path}?{urllib.parse.urlencode(params)}", headers={"User-Agent": USER_AGENT})
    return json.loads(urllib.request.urlopen(req, timeout=60).read()), public

def restore_abstract(inverted, max_words=400):
    words = sorted((pos, word) for word, positions in (inverted or {}).items() for pos in positions)
    return " ".join(word for _, word in words[:max_words])

try:
    w, request = openalex_get("/works/" + docs[0]["id"], select="display_name,abstract_inverted_index")
    inv = w["abstract_inverted_index"]
    print("запрос:", request)
    print("в индексе:", dict(list(inv.items())[:6]), "…")
    print("восстановлено:", restore_abstract(inv)[:160], "…")
except Exception as e:
    print("сеть недоступна, раздел пропущен:", type(e).__name__, e)

сеть недоступна, раздел пропущен: NameError name 'docs' is not defined


## 2. Нарезка на фрагменты

Длинный документ в окно контекста не кладут целиком: его режут на фрагменты и ищут среди
них. Параметры нарезки — размер фрагмента и перекрытие соседних фрагментов.
Фрагмент, вырезанный из середины, теряет сведения о документе: о чём статья, кто автор.
Простой способ вернуть контекст — приписать к фрагменту название документа.

In [4]:
def chunk_words(text, size=60, overlap=15):
    words = text.split()
    step = size - overlap
    return [" ".join(words[i:i + size]) for i in range(0, max(1, len(words) - overlap), step)]

chunks = []                                    # (номер документа, текст фрагмента)
for i, d in enumerate(docs):
    for c in chunk_words(d["abstract"]):
        chunks.append((i, c))
print("фрагментов:", len(chunks), "| на документ в среднем", round(len(chunks) / len(docs), 1))
i, c = chunks[2]
print("\nФрагмент как он есть:\n ", c)
print("\nФрагмент с названием документа:\n ", docs[i]["title"] + " — " + c)

NameError: name 'docs' is not defined

## 3. Мешок слов и инвертированный индекс

Лексический поиск сравнивает слова. Текст превращается в набор слов без порядка
(«мешок слов»); индекс для каждого слова хранит документы, где оно встречается, и число
вхождений. Поиск читает только списки слов запроса и не просматривает всю коллекцию.

In [5]:
STOP = set("a an and are as at be by for from has have in is it its of on or that the this to was were which with we our can".split())

def tokenize(text):
    return [w for w in re.findall(r"[a-zа-яё0-9]+", text.lower()) if w not in STOP and len(w) > 1]

index = defaultdict(dict)                      # слово -> {номер документа: число вхождений}
doc_len = []
for i, d in enumerate(docs):
    tokens = tokenize(d["text"])
    doc_len.append(len(tokens))
    for word, tf in Counter(tokens).items():
        index[word][i] = tf
n_docs, avg_len = len(docs), sum(doc_len) / len(docs)
print("слов в словаре:", len(index), "| средняя длина документа:", round(avg_len), "слов")
for word in ("arrhenius", "retrieval", "sparql", "temperature"):
    postings = index[word]
    print(f"  {word:12s} встречается в {len(postings):2d} документах; первые записи: {dict(list(postings.items())[:5])}")

NameError: name 'docs' is not defined

## 4. TF-IDF

Вес слова $w$ в документе $d$ — произведение двух множителей:

$$\mathrm{tfidf}(w, d) = \mathrm{tf}(w, d)\cdot \mathrm{idf}(w), \qquad \mathrm{idf}(w) = \ln\frac{|D|}{\mathrm{df}(w)},$$

где $\mathrm{tf}(w,d)$ — число вхождений слова в документ, $\mathrm{df}(w)$ — число документов
со словом, $|D|$ — число документов. Слово, которое есть почти везде, получает малый вес.

Документ и запрос становятся векторами таких весов: координата — слово словаря. Оценка
документа по запросу $q$ — косинус угла между двумя векторами:

$$s(q, d) = \frac{\sum_{w} \mathrm{tfidf}(w, q)\,\mathrm{tfidf}(w, d)}{\|q\|\,\|d\|}.$$

In [6]:
idf = {w: math.log(n_docs / len(p)) for w, p in index.items()}
doc_norm = [0.0] * n_docs
for w, postings in index.items():
    for i, tf in postings.items():
        doc_norm[i] += (tf * idf[w]) ** 2
doc_norm = [math.sqrt(x) for x in doc_norm]

def tfidf_scores(query):
    q_tf = Counter(w for w in tokenize(query) if w in idf)
    q_vec = {w: tf * idf[w] for w, tf in q_tf.items()}        # вектор запроса: веса его слов
    q_norm = math.sqrt(sum(v * v for v in q_vec.values()))
    scores = np.zeros(n_docs)
    if not q_norm:
        return scores
    for w, q_weight in q_vec.items():
        for i, tf in index[w].items():
            scores[i] += q_weight * tf * idf[w] / (q_norm * doc_norm[i])     # косинус: скалярное произведение, делённое на длины
    return scores

def top(scores, k=5):
    order = np.argsort(-scores)[:k]
    return [int(i) for i in order if scores[i] > 0]

def show(query, ranked, scores=None, k=5):
    print(f"Запрос: {query}")
    if not ranked:
        print("   ничего не найдено")
    for r, i in enumerate(ranked[:k], 1):
        s = f"{scores[i]:6.3f} " if scores is not None else ""
        print(f"   {r}. {s}[{docs[i]['topic']}] {docs[i]['title'][:80]}")

q = "activation energy of chemical reactions"
s = tfidf_scores(q)
show(q, top(s), s)

NameError: name 'n_docs' is not defined

## 5. BM25

BM25 исправляет два свойства TF-IDF. Первое: десятое вхождение слова значит меньше, чем
первое, — вклад частоты **насыщается**. Второе: длинный документ содержит больше слов,
поэтому частота **нормируется на длину**:

$$\mathrm{BM25}(q, d) = \sum_{w \in q} \mathrm{idf}(w)\cdot
\frac{\mathrm{tf}(w,d)\,(k_1 + 1)}{\mathrm{tf}(w,d) + k_1\left(1 - b + b\,\frac{|d|}{\overline{|d|}}\right)},
\qquad \mathrm{idf}(w) = \ln\left(1 + \frac{|D| - \mathrm{df}(w) + 0.5}{\mathrm{df}(w) + 0.5}\right).$$

$k_1$ задаёт скорость насыщения (обычно 1,2–2), $b$ — силу нормировки на длину (0 — нет,
1 — полная; обычно 0,75), $|d|$ — длина документа, $\overline{|d|}$ — средняя длина.

In [7]:
def bm25_scores(query, k1=1.2, b=0.75):
    scores = np.zeros(n_docs)
    for w in tokenize(query):
        postings = index.get(w, {})
        w_idf = math.log(1 + (n_docs - len(postings) + 0.5) / (len(postings) + 0.5))
        for i, tf in postings.items():
            scores[i] += w_idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * doc_len[i] / avg_len))
    return scores

q = "activation energy of chemical reactions"
s = bm25_scores(q)
show(q, top(s), s)

NameError: name 'n_docs' is not defined

### Опыт: один запрос, две лексические модели

Обе модели ранжируют документы по словам запроса. Вопрос опыта: какой документ окажется
выше — содержащий все слова запроса по разу или одно слово запроса много раз.

- **Коллекция:** учебная библиотека, 80 документов (название и аннотация).
- **Запрос:** `activation energy of chemical reactions`; после отбрасывания служебных слов
  в нём четыре слова. Слова не приводятся к основе: `reaction` и `reactions` — разные слова.
- **Модели:** TF-IDF с косинусом (раздел 4) и BM25 с $k_1 = 1{,}2$, $b = 0{,}75$.
- **Документ А** — работа по химической кинетике, в которой есть все четыре слова запроса.
  **Документ Б** — работа об онтологии химических веществ: слово `chemical` встречается в
  ней 12 раз, остальных слов запроса нет.
- **Что измеряем:** место каждого документа в двух выдачах и вклад каждого слова запроса
  в оценку.

Ожидание: документ А отвечает запросу, документ Б — нет.

In [8]:
q = "activation energy of chemical reactions"
q_words = tokenize(q)
st, sb = tfidf_scores(q), bm25_scores(q)
doc_b = int(np.argsort(-st)[0])                                   # документ Б: первое место TF-IDF
doc_a = next(int(i) for i in np.argsort(-sb) if docs[i]["topic"] == "химическая кинетика")   # документ А
place = lambda scores, i: int(np.where(np.argsort(-scores) == i)[0][0]) + 1

print("слово запроса:   ", "".join(f"{w:>12s}" for w in q_words))
print("документов с ним:", "".join(f"{len(index.get(w, {})):12d}" for w in q_words), f"  (из {n_docs})")
for name, i in (("А", doc_a), ("Б", doc_b)):
    print(f"\nДокумент {name}: [{docs[i]['topic']}] {docs[i]['title'][:75]} | {doc_len[i]} слов")
    print("  вхождений:     ", "".join(f"{index.get(w, {}).get(i, 0):12d}" for w in q_words))
    print(f"  TF-IDF: оценка {st[i]:.2f}, место {place(st, i)} | BM25: оценка {sb[i]:.2f}, место {place(sb, i)}")

def tfidf_parts(i):       # вклад каждого слова запроса в косинус
    q_norm = math.sqrt(sum(idf[w] ** 2 for w in q_words))
    return [idf[w] * index[w].get(i, 0) * idf[w] / (q_norm * doc_norm[i]) for w in q_words]

def bm25_parts(i, k1=1.2, b=0.75):
    parts = []
    for w in q_words:
        tf, df = index[w].get(i, 0), len(index[w])
        w_idf = math.log(1 + (n_docs - df + 0.5) / (df + 0.5))
        parts.append(w_idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * doc_len[i] / avg_len)))
    return parts

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
labels = ["Б: онтология\nхимических веществ", "А: кинетика\nреакций"]
for a, parts, title in ((ax[0], tfidf_parts, "TF-IDF: вклад слов запроса"), (ax[1], bm25_parts, "BM25: вклад слов запроса")):
    bottom = np.zeros(2)
    for j, (w, color) in enumerate(zip(q_words, ["#3B5BA9", "#2E8B57", "#C0392B", "#E69F00"])):
        vals = np.array([parts(doc_b)[j], parts(doc_a)[j]])
        a.bar(labels, vals, bottom=bottom, color=color, width=0.5, label=w, edgecolor="white")
        bottom += vals
    a.set_title(title)
ax[0].legend(frameon=False, fontsize=7)
fig.tight_layout(); plt.show()

NameError: name 'tfidf_scores' is not defined

**Что видно.** У TF-IDF документ Б стоит первым: вес слова растёт линейно с числом
вхождений, а редкое в коллекции слово получает большой $\mathrm{idf}$, который входит в
косинус дважды — в вес слова запроса и в вес слова документа. Оценка не отличает «одно
слово много раз» от «все слова запроса по разу». BM25 ограничивает вклад повторов:
документ Б опускается, документ А поднимается.

In [9]:
tf = np.arange(0, 21)
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(tf, tf / 4, color="#999999", ls="--", label="без насыщения (TF, масштаб 1:4)")
for k1, color in ((0.5, "#C0392B"), (1.2, "#3B5BA9"), (2.0, "#2E8B57")):
    ax[0].plot(tf, tf * (k1 + 1) / (tf + k1), color=color, label=f"BM25, $k_1$ = {k1}")
ax[0].set_xlabel("число вхождений слова в документ"); ax[0].set_ylabel("вклад слова"); ax[0].set_ylim(0, 5)
ax[0].set_title("Насыщение по частоте"); ax[0].legend(frameon=False, fontsize=7)
ratio = np.linspace(0.25, 4, 50)
for b, color in ((0.0, "#999999"), (0.75, "#3B5BA9"), (1.0, "#C0392B")):
    ax[1].plot(ratio, 3 * 2.2 / (3 + 1.2 * (1 - b + b * ratio)), color=color, label=f"$b$ = {b}")
ax[1].set_xlabel("длина документа / средняя длина"); ax[1].set_ylabel("вклад слова при tf = 3")
ax[1].set_title("Нормировка на длину"); ax[1].legend(frameon=False, fontsize=7)
fig.tight_layout(); fig.savefig("fig_bm25_saturation.png", dpi=160); plt.show()

NameError: name 'plt' is not defined

## 6. Векторный поиск

Кодировщик — нейросеть семейства BERT — переводит текст в вектор из $m$ чисел. Близость
текстов измеряется косинусом угла между векторами. Кодировщик один: сначала через него
проходят все документы библиотеки, затем — каждый запрос. Векторы документов считаются
один раз, при поиске кодируется только запрос.

Кодировщик обучен на парах «запрос — подходящий текст». Модель E5 различает две роли
приставками: `query: ` перед запросом и `passage: ` перед документом.

In [10]:
from sentence_transformers import SentenceTransformer

t0 = time.time()
encoder = SentenceTransformer(MODEL_NAME, device="cpu")
E5 = "e5" in MODEL_NAME.lower()

def encode(texts, role):
    prefix = {"query": "query: ", "passage": "passage: "}[role] if E5 else ""
    return encoder.encode([prefix + t for t in texts], normalize_embeddings=True, show_progress_bar=False)

doc_vecs = encode([d["text"] for d in docs], "passage")
print("матрица векторов документов:", doc_vecs.shape, "| время:", round(time.time() - t0, 1), "с")

def vec_scores(query):
    return doc_vecs @ encode([query], "query")[0]       # векторы нормированы: скалярное произведение = косинус

q = "activation energy of chemical reactions"
s = vec_scores(q)
show(q, top(s), s)

C:\Users\yury_\Shad_projects\HOM_project_new\.venv-course\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'MODEL_NAME' is not defined

На запросе выше первые места трёх моделей разные: TF-IDF поставил первой работу об
онтологии химических веществ (слово chemical повторено в ней 12 раз), BM25 — работу о
влиянии температуры на скорость обмена веществ, векторный поиск — работы об энергии
активации.

Три запроса ниже показывают, где модели расходятся: точный термин, пересказ без общих
слов с документами и запрос на другом языке.

In [11]:
for q in ["Eyring equation transition state",
          "why do processes speed up when it gets hotter",
          "почему реакции идут быстрее при нагревании"]:
    sb, sv = bm25_scores(q), vec_scores(q)
    print("=" * 100); show(q + "   — BM25", top(sb, 3), sb, 3); show(q + "   — вектор", top(sv, 3), sv, 3)

NameError: name 'n_docs' is not defined

**Фрагменты против документов.** Поиск по фрагментам находит место в тексте. Ниже —
лучший фрагмент для запроса и документ, из которого он вырезан.

In [12]:
chunk_vecs = encode([c for _, c in chunks], "passage")
q = "why do processes speed up when it gets hotter"
s = chunk_vecs @ encode([q], "query")[0]
for j in np.argsort(-s)[:3]:
    i, c = chunks[j]
    print(f"{s[j]:.3f}  документ: {docs[i]['title'][:70]}\n       фрагмент: {c[:230]} …\n")

NameError: name 'encode' is not defined

## 7. Проекция векторов на плоскость

384 измерения нельзя нарисовать, поэтому векторы проецируют на плоскость с сохранением
соседства (UMAP). Слева точки раскрашены темой, по которой работа попала в библиотеку,
справа — кластером, найденным методом $k$ средних без сведений о темах.

In [13]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
try:
    import umap
    xy = umap.UMAP(n_neighbors=10, min_dist=0.3, metric="cosine", random_state=0).fit_transform(doc_vecs)
    method = "UMAP"
except Exception:
    from sklearn.manifold import TSNE
    xy = TSNE(perplexity=15, metric="cosine", init="random", random_state=0).fit_transform(doc_vecs)
    method = "t-SNE"

clusters = KMeans(n_clusters=len(topics), n_init=10, random_state=0).fit_predict(doc_vecs)
topic_id = np.array([topics.index(d["topic"]) for d in docs])
palette = ["#3B5BA9", "#C0392B", "#2E8B57", "#E69F00", "#8E44AD"]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
for t, name in enumerate(topics):
    m = topic_id == t
    ax[0].scatter(xy[m, 0], xy[m, 1], s=46, color=palette[t], label=name, edgecolor="white", linewidth=0.6)
ax[0].set_title(f"Цвет — тема запроса ({method})", fontsize=11); ax[0].legend(frameon=False, fontsize=9)
for c in range(len(topics)):
    m = clusters == c
    ax[1].scatter(xy[m, 0], xy[m, 1], s=46, color=palette[c], edgecolor="white", linewidth=0.6)
ax[1].set_title("Цвет — кластер $k$ средних", fontsize=11)
for a in ax:
    a.set_xticks([]); a.set_yticks([])
fig.tight_layout(); fig.savefig("fig_projection.png", dpi=160); plt.show()
print("совпадение кластеров с темами (скорректированный индекс Рэнда, 1 — полное):", round(adjusted_rand_score(topic_id, clusters), 2))

NameError: name 'doc_vecs' is not defined

## 8. Сначала кластер, потом поиск

Точный поиск сравнивает запрос с каждым вектором: $|D|$ сравнений. Если векторы заранее
разбить на $c$ кластеров, запрос сравнивается с $c$ центрами и с векторами ближайших
кластеров. При $c \approx \sqrt{|D|}$ и одном просматриваемом кластере это около
$2\sqrt{|D|}$ сравнений. Цена — часть нужных векторов остаётся в соседних кластерах.
Ниже замер на фрагментах библиотеки: доля точной выдачи top-5, найденная через кластеры.

In [14]:
n = len(chunk_vecs)
n_list = int(round(math.sqrt(n)))
km = KMeans(n_clusters=n_list, n_init=5, random_state=0).fit(chunk_vecs)
members = [np.where(km.labels_ == c)[0] for c in range(n_list)]
queries = encode([d["title"] for d in docs], "query")      # 80 запросов: названия работ

print(f"фрагментов: {n}, кластеров: {n_list}")
print("кластеров просмотрено | сравнений на запрос | доля точного top-5")
exact = [set(np.argsort(-(chunk_vecs @ qv))[:5]) for qv in queries]
print(f"{'все (точный поиск)':>21} | {n:19d} | 1.00")
for n_probe in (1, 2, 4):
    found, cost = 0, 0
    for qv, ex in zip(queries, exact):
        near = np.argsort(-(km.cluster_centers_ @ qv))[:n_probe]
        cand = np.concatenate([members[c] for c in near])
        cost += n_list + len(cand)
        got = set(cand[np.argsort(-(chunk_vecs[cand] @ qv))[:5]])
        found += len(got & ex)
    print(f"{n_probe:21d} | {cost // len(queries):19d} | {found / (5 * len(queries)):.2f}")

NameError: name 'chunk_vecs' is not defined

## 9. Как измерить качество

Нужны запросы с известными правильными ответами. Здесь разметка грубая: документ считается
подходящим, если он из темы запроса. Метрики для выдачи длины $k$:

- **точность@k** — доля подходящих среди первых $k$;
- **полнота@k** — доля всех подходящих, попавшая в первые $k$;
- **MRR** — среднее по запросам от $1/\text{место первого подходящего}$;
- **nDCG@k** — сумма $1/\log_2(\text{место}+1)$ по подходящим документам, делённая на
  значение для идеальной выдачи.

**Слияние выдач (RRF).** Гибридный поиск объединяет списки BM25 и векторного поиска:
оценка документа — сумма $1/(60 + \text{место})$ по спискам.

In [15]:
EVAL = [  # запрос, тема, для которой документы считаются подходящими
    ("activation energy temperature dependence Arrhenius rate constant", "химическая кинетика"),
    ("why does a warmer reaction mixture transform faster", "химическая кинетика"),
    ("metabolic rate of ectotherms and body temperature", "температура в биологии"),
    ("enzymes of cold-adapted organisms", "температура в биологии"),
    ("BM25 ranking function", "информационный поиск"),
    ("finding relevant passages with neural encoders", "информационный поиск"),
    ("SPARQL endpoint linked data", "графы знаний"),
    ("machine-readable description of concepts and relations", "графы знаний"),
    ("скорость реакции и температура", "химическая кинетика"),
    ("поиск документов по запросу", "информационный поиск"),
]

def ranking(scores, k=20):
    return [int(i) for i in np.argsort(-scores)[:k] if scores[i] > 0]

def rrf(*rankings, c=60):
    fused = defaultdict(float)
    for r in rankings:
        for place, i in enumerate(r, 1):
            fused[i] += 1 / (c + place)
    return sorted(fused, key=fused.get, reverse=True)

METHODS = {
    "TF-IDF": lambda q: ranking(tfidf_scores(q)),
    "BM25": lambda q: ranking(bm25_scores(q)),
    "вектор": lambda q: ranking(vec_scores(q)),
    "гибрид (RRF)": lambda q: rrf(ranking(bm25_scores(q)), ranking(vec_scores(q))),
}

def metrics(ranked, relevant, k=10):
    hits = [1 if i in relevant else 0 for i in ranked[:k]]
    first = next((r for r, h in enumerate(hits, 1) if h), None)
    dcg = sum(h / math.log2(r + 1) for r, h in enumerate(hits, 1))
    ideal = sum(1 / math.log2(r + 1) for r in range(1, min(k, len(relevant)) + 1))
    return {"точность@5": sum(hits[:5]) / 5, f"полнота@{k}": sum(hits) / len(relevant),
            "MRR": 1 / first if first else 0.0, f"nDCG@{k}": dcg / ideal}

def evaluate(eval_set, methods=METHODS):
    table = {}
    for name, search in methods.items():
        rows = [metrics(search(q), {i for i, d in enumerate(docs) if d["topic"] == t}) for q, t in eval_set]
        table[name] = {m: sum(r[m] for r in rows) / len(rows) for m in rows[0]}
    return table

def print_table(table, title):
    cols = list(next(iter(table.values())))
    print(title); print(f"{'':14s}" + "".join(f"{c:>12s}" for c in cols))
    for name, row in table.items():
        print(f"{name:14s}" + "".join(f"{row[c]:12.2f}" for c in cols))

print_table(evaluate(EVAL[:8]), "Запросы на английском (8)"); print()
print_table(evaluate(EVAL[8:]), "Запросы на русском к английской библиотеке (2)")

NameError: name 'tfidf_scores' is not defined

### Два изменённых запроса (шаг 1.3 практики)
Ниже предложены две новые формулировки по прежним темам. Прочитайте их, при желании замените своими.
Сохраняем сравнение исходных и изменённых выдач; ничего не заполняется до запуска.

In [16]:
EVAL_CHANGED = list(EVAL)
EVAL_CHANGED[0] = ("Arrhenius activation barrier reaction rate temperature", EVAL[0][1])
EVAL_CHANGED[1] = ("What makes a chemical transformation proceed more quickly when heated?", EVAL[1][1])
original_scores = evaluate(EVAL)
changed_scores = evaluate(EVAL_CHANGED)
print_table(original_scores, "Исходные запросы")
print_table(changed_scores, "Изменены две формулировки")
(PROJECT / "reports").mkdir(exist_ok=True)
(PROJECT / "reports/search_query_comparison.json").write_text(json.dumps({
 "original_queries": EVAL, "changed_queries": EVAL_CHANGED,
 "original_scores": original_scores, "changed_scores": changed_scores}, ensure_ascii=False, indent=2), encoding="utf-8")

NameError: name 'tfidf_scores' is not defined

**Как читать таблицу.** В каждой теме 20 документов, поэтому полнота@10 не превышает 0,5.
Лексические модели сильны там, где запрос повторяет слова
документов, и дают ноль на запросе на другом языке: общих слов нет. Векторный поиск
находит пересказ и перевод, но может пропустить редкий точный термин. Гибрид берёт лучшее
из двух списков. Числа относятся к этой библиотеке и этой разметке: на другой коллекции
порядок моделей бывает иным, поэтому сравнение проводят на своих запросах.

Поменяйте запросы в списке `EVAL` или параметры `k1` и `b` и выполните ячейку ещё раз.

## 10. Переранжирование

Кодировщик из раздела 6 сжимает документ в один вектор, не зная запроса. **Совместный
кодировщик** получает запрос и документ одной последовательностью и оценивает пару
целиком: это точнее и на порядки медленнее. Поэтому его применяют ко второму шагу —
к первым десяткам кандидатов первичной выдачи.

In [17]:
RUN_RERANK = True      # загрузка ещё одной модели, около 90 МБ; работает с английскими текстами
q = "does the reaction rate always follow the Arrhenius law"
first = ranking(bm25_scores(q), 10)
show(q + "   — BM25, первые 5", first, k=5)
if RUN_RERANK:
    try:
        from sentence_transformers import CrossEncoder
        t0 = time.time()
        reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
        pair_scores = reranker.predict([(q, docs[i]["text"]) for i in first])
        second = [first[j] for j in np.argsort(-pair_scores)]
        print(); show(q + "   — после переранжирования", second, k=5)
        print(f"\nоценено пар: {len(first)}; время с загрузкой модели: {time.time() - t0:.1f} с")
        print("перестановка мест (было -> стало):", {r: second.index(i) + 1 for r, i in enumerate(first[:5], 1)})
    except Exception as e:
        print("переранжирование пропущено:", type(e).__name__, str(e)[:120])

NameError: name 'n_docs' is not defined

## 11. Внешний граф знаний: Wikidata и SPARQL

Wikidata хранит утверждения тройками «субъект — предикат — объект». У сущностей
идентификаторы вида `Q507505` (уравнение Аррениуса), у свойств — `P138` («названо в
честь»). Запрос SPARQL — шаблон троек с переменными: служба возвращает все подстановки,
при которых тройки шаблона есть в графе.

Служба запросов ограничивает частоту обращений: в заголовке `User-Agent` указывается, кто
спрашивает, между запросами делается пауза. При ответе 429 функция ниже ждёт указанное
службой время и повторяет запрос один раз; в дни перегрузки служба пропускает один запрос
в минуту.

In [18]:
WDQS = "https://query.wikidata.org/sparql"

def sparql(query, pause=2.0):
    url = WDQS + "?" + urllib.parse.urlencode({"query": query, "format": "json"})
    req = urllib.request.Request(url, headers={"User-Agent": USER_AGENT, "Accept": "application/sparql-results+json"})
    for attempt in (1, 2):
        try:
            rows = json.loads(urllib.request.urlopen(req, timeout=60).read())["results"]["bindings"]
            break
        except urllib.error.HTTPError as e:
            if e.code != 429 or attempt == 2:
                raise
            wait = min(int(e.headers.get("Retry-After") or 60), 90) + 2       # служба просит подождать
            print(f"   служба ограничила частоту запросов, жду {wait} с …")
            time.sleep(wait)
    time.sleep(pause)
    return [{k: v["value"].replace("http://www.wikidata.org/entity/", "") for k, v in r.items()} for r in rows]

Q1 = '''
SELECT ?item ?itemLabel ?classLabel WHERE {
  ?item wdt:P138 wd:Q80956 .                # ?item названо в честь Сванте Аррениуса
  OPTIONAL { ?item wdt:P31 ?class . }       # ?item — частный случай ?class
  SERVICE wikibase:label { bd:serviceParam wikibase:language "ru,en". }
}'''
try:
    rows = sparql(Q1)
    print("Что названо в честь Аррениуса:", len(rows), "строк")
    for r in rows:
        print(f"   {r['item']:10s} {r.get('itemLabel', ''):38s} {r.get('classLabel', '')}")
    wikidata_record = {"source": "Wikidata Query Service", "request": WDQS, "sparql": Q1.strip(),
                       "retrieved": date.today().isoformat(), "rows": len(rows)}
except Exception as e:
    print("сеть недоступна или служба ограничила запросы:", type(e).__name__, e)

   служба ограничила частоту запросов, жду 92 с …


Что названо в честь Аррениуса: 10 строк
   Q507505    Уравнение Аррениуса                    уравнение
   Q684778    (5697) Arrhenius                       астероид
   Q695171    аррениусовский график                  график
   Q699618    Аррениус                               лунный кратер
   Q1186871   Аррениус                               ударный кратер
   Q7448675   Arrhenius acid–base theory             теория
   Q7448675   Arrhenius acid–base theory             научная теория
   Q10949067  Arrheniusfjellet                       гора
   Q105415606 Arrhenius number                       
   Q131177097 Arrhenius Plaque                       награда


Текстовый поиск такой список не гарантирует: слова «Аррениус» может не быть в описании
кратера или астероида. Запрос по связям возвращает все сущности, у которых связь записана
в графе, вместе с идентификаторами.

Второй запрос читает уточнитель утверждения — дату присуждения награды: лауреаты
Нобелевской премии по химии до 1920 года. Аррениус получил её в 1903 году.

In [19]:
Q2 = '''
SELECT ?person ?personLabel ?year WHERE {
  ?person p:P166 ?award .                       # у ?person есть награда
  ?award ps:P166 wd:Q44585 ; pq:P585 ?date .    # Нобелевская премия по химии, с датой присуждения
  BIND(YEAR(?date) AS ?year)
  FILTER(?year < 1920)
  SERVICE wikibase:label { bd:serviceParam wikibase:language "ru,en". }
} ORDER BY ?year'''
try:
    for r in sparql(Q2):
        print(f"   {r['year']}  {r['person']:10s} {r.get('personLabel', '')}")
except Exception as e:
    print("сеть недоступна или служба ограничила запросы:", type(e).__name__, e)

   служба ограничила частоту запросов, жду 92 с …


   1901  Q102822    Якоб Вант-Гофф
   1902  Q70554     Герман Эмиль Фишер
   1903  Q80956     Сванте Август Аррениус
   1904  Q950726    Уильям Рамзай
   1905  Q57078     Адольф Байер
   1906  Q102804    Анри Муассан
   1907  Q43917     Эдуард Бухнер
   1908  Q9123      Эрнест Резерфорд
   1909  Q12658     Вильгельм Фридрих Оствальд
   1910  Q57127     Отто Валлах
   1911  Q7186      Мария Склодовская-Кюри
   1912  Q104575    Поль Сабатье
   1912  Q104582    Виктор Гриньяр
   1913  Q123014    Альфред Вернер
   1914  Q189465    Теодор Уильям Ричардс
   1915  Q77072     Рихард Вильштеттер
   1918  Q57075     Фриц Габер


С 2025 года сведения о научных статьях Wikidata вынесены в отдельную точку доступа; в
основной их нет. Для поиска статей в курсе используется OpenAlex.

## 12. Научный граф знаний: OpenAlex

OpenAlex — открытый индекс научных работ: работы, авторы, источники, организации, темы и
связи между ними (цитирования, авторство, принадлежность). Доступ — запросами по HTTP:
`search` — поиск по тексту, `filter` — отбор по полям, `group_by` — подсчёт по группам.
Без ключа сервис отвечает, но суточный лимит меньше; бесплатный ключ выдаётся после
регистрации на openalex.org.

In [20]:
try:
    data, request = openalex_get("/works", per_page=1)
    print("работ в OpenAlex:", f"{data['meta']['count']:,}".replace(",", " "), "| дата:", date.today().isoformat())

    data, request = openalex_get("/works", search="Arrhenius equation", sort="cited_by_count:desc", per_page=5,
                                 filter="type:review,from_publication_date:2015-01-01",
                                 select="id,display_name,publication_year,cited_by_count,open_access")
    print("\nСамые цитируемые обзоры с 2015 года по запросу «Arrhenius equation»; найдено:", data["meta"]["count"])
    for w in data["results"]:
        print(f"   {w['publication_year']}  цит. {w['cited_by_count']:5d}  {w['open_access']['oa_status']:7s} {w['display_name'][:75]}")
    print("запрос:", request)
except Exception as e:
    print("сеть недоступна:", type(e).__name__, e)

работ в OpenAlex: 328 192 191 | дата: 2026-10-20



Самые цитируемые обзоры с 2015 года по запросу «Arrhenius equation»; найдено: 273
   2018  цит.  1427  green   Energy transfer catalysis mediated by visible light: principles, applicatio
   2021  цит.  1264  green   Dynamic interactions at the mineral–organic matter interface
   2021  цит.  1204  green   Electrochemical impedance spectroscopy
   2020  цит.  1017  green   Single-Atom Alloy Catalysis
   2016  цит.   904  hybrid  Water: A Tale of Two Liquids
запрос: https://api.openalex.org/works?search=Arrhenius+equation&sort=cited_by_count%3Adesc&per_page=5&filter=type%3Areview%2Cfrom_publication_date%3A2015-01-01&select=id%2Cdisplay_name%2Cpublication_year%2Ccited_by_count%2Copen_access


In [21]:
LAIDLER = "W2052953969"      # K. J. Laidler, The development of the Arrhenius equation, 1984
try:
    work, request = openalex_get("/works/" + LAIDLER)
    record = {"source": "OpenAlex", "request": request, "retrieved": date.today().isoformat(),
              "openalex_id": LAIDLER, "doi": work["doi"], "title": work["display_name"],
              "year": work["publication_year"], "cited_by": work["cited_by_count"],
              "oa_status": work["open_access"]["oa_status"], "pages": [work["biblio"]["first_page"], work["biblio"]["last_page"]]}
    print("Запись о происхождении:"); print(json.dumps(record, ensure_ascii=False, indent=1))
    print("\nВ печатной статье страницы 494–498. Метаданные внешнего графа сверяются с источником.")

    data, _ = openalex_get("/works", filter=f"cites:{LAIDLER}", group_by="publication_year")
    by_year = sorted((int(g["key"]), g["count"]) for g in data["group_by"] if g["key"] and int(g["key"]) >= 1985)
    fig, ax = plt.subplots(figsize=(8, 2.6))
    ax.bar([y for y, _ in by_year], [c for _, c in by_year], color="#3B5BA9")
    ax.set_ylabel("работ в год"); ax.set_title("Кто ссылается на Laidler (1984): group_by=publication_year")
    fig.tight_layout(); fig.savefig("fig_openalex_citations.png", dpi=160); plt.show()
    data, _ = openalex_get("/works", filter=f"cites:{LAIDLER}", group_by="primary_topic.field.id")
    print("Области знаний ссылающихся работ:")
    for g in data["group_by"][:8]:
        print(f"   {g['count']:5d}  {g['key_display_name']}")
except Exception as e:
    print("сеть недоступна:", type(e).__name__, e)

Запись о происхождении:
{
 "source": "OpenAlex",
 "request": "https://api.openalex.org/works/W2052953969",
 "retrieved": "2026-10-20",
 "openalex_id": "W2052953969",
 "doi": "https://doi.org/10.1021/ed061p494",
 "title": "The development of the Arrhenius equation",
 "year": 1984,
 "cited_by": 1702,
 "oa_status": "closed",
 "pages": [
  "494",
  "494"
 ]
}

В печатной статье страницы 494–498. Метаданные внешнего графа сверяются с источником.


сеть недоступна: NameError name 'plt' is not defined


## 13. Запросы как инструменты MCP

До сих пор запросы писали мы. Чтобы ими пользовался агент, каждый запрос оформляется
**инструментом**: имя, описание и схема аргументов. Модель видит только эти три вещи и по
ним решает, когда вызвать инструмент и с какими аргументами. Файл `research_mcp.py` в этой
папке — сервер MCP с пятью инструментами. Аргументы с выбором из списка (`work_type`,
`open_access`, `sort`, `relation`, `language`) объявлены перечислениями: модель не может
подставить в них произвольный текст.

In [22]:
try:
    import research_mcp
    tools = await research_mcp.mcp.list_tools()
    for t in tools:
        schema = getattr(t, "input_schema", None) or getattr(t, "inputSchema", {})   # имя поля зависит от версии mcp
        props = schema.get("properties", {})
        enums = {k: v["enum"] for k, v in props.items() if "enum" in v}
        print(f"{t.name}({', '.join(props)})")
        print("   " + t.description.strip().split("\n")[0])
        for k, v in enums.items():
            print(f"   {k} — одно из: {v}")
except Exception as e:
    print("инструменты модуля недоступны:", type(e).__name__, e)
    print("нужны файл research_mcp.py рядом с ноутбуком и пакет mcp в окружении ядра: %pip install mcp")

инструменты модуля недоступны: ModuleNotFoundError No module named 'research_mcp'
нужны файл research_mcp.py рядом с ноутбуком и пакет mcp в окружении ядра: %pip install mcp


In [23]:
try:
    answer = json.loads(research_mcp.openalex_links("W2052953969", relation="cited_by", sort="cited_by_count", limit=3))
    print(json.dumps(answer, ensure_ascii=False, indent=1)[:1500])
except Exception as e:
    print("сеть недоступна или модуль не загружен:", type(e).__name__, e)

сеть недоступна или модуль не загружен: NameError name 'research_mcp' is not defined


Так выглядит ответ, который попадает в окно контекста агента: записи и поле `provenance` —
откуда и когда получены данные. Поля `title` и `abstract` помечены как внешний текст: это
данные, инструкции из них агент не выполняет.

**Как подключить модуль к учебному агенту** — файл `CONNECT.md` в этой папке: модуль
копируется в `src/` агента, рядом со встроенным сервером, запускающий файл — в `bin/`,
в `recipes/ai4science.yaml` добавляются шесть строк, в `.env` — ключ. После подключения
агент вызывает инструменты сам:

```
▸ openalex_work research-mcp
    work: 10.1021/ed061p494
▸ wikidata_search research-mcp
    text: уравнение Аррениуса
Журнал: Journal of Chemical Education; год: 1984; OpenAlex: W2052953969.
Wikidata для «уравнение Аррениуса»: Q507505.
```

## 14. Своя тема

Замените значения двух переменных и выполните ячейку. Она соберёт из OpenAlex библиотеку
по вашей теме (20 работ с аннотациями), сравнит BM25 и векторный поиск на ваших запросах
и запишет файл `my_search_report.md`: таблица первых мест выдачи и запись о происхождении.
Запросы к OpenAlex — на английском; запросы к своей библиотеке — на любом языке.

In [24]:
MY_TOPIC = "quantum dot photon indistinguishability"                 # например: "perovskite solar cell degradation"
MY_QUERIES = [
    "Hong Ou Mandel interference semiconductor quantum dot photons",
    "How does emission time jitter change two photon interference?",
    "Как отличить чистое дефазирование от временного джиттера фотонов?",
    "Can a delay scan determine the radiative lifetime without time resolved detection?",
    "multiphoton emission correction Hong Ou Mandel visibility",
]               # пять запросов к собранной библиотеке, например: ["что ускоряет деградацию", ...]

if MY_TOPIC and MY_QUERIES:
    (PROJECT / "reports").mkdir(exist_ok=True)
    (PROJECT / "library_search").mkdir(exist_ok=True)
    for target in (PROJECT / "reports/my_search_report.md", PROJECT / "library_search/my_library.json"):
        if target.exists():
            backup = target.with_name(target.name + "." + str(time.time_ns()) + ".bak")
            backup.write_bytes(target.read_bytes())
    data, request = openalex_get("/works", search=MY_TOPIC, per_page=50, filter="has_abstract:true",
                                 select="id,doi,display_name,publication_year,cited_by_count,abstract_inverted_index")
    mine = [{"id": w["id"].rsplit("/", 1)[-1], "title": w["display_name"], "year": w["publication_year"],
             "doi": w["doi"], "text": (w["display_name"] or "") + ". " + restore_abstract(w["abstract_inverted_index"])}
            for w in data["results"] if len(w.get("abstract_inverted_index") or {}) > 40][:20]
    if len(mine) < 20:
        raise ValueError(f"Найдено только {len(mine)} работ с достаточными аннотациями; расширьте MY_TOPIC и повторите только эту ячейку.")
    (PROJECT / "library_search/my_library.json").write_text(json.dumps({
        "source": "OpenAlex", "request": request, "date": date.today().isoformat(),
        "topic": MY_TOPIC, "documents": mine
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    my_index, my_len = defaultdict(dict), []
    for i, d in enumerate(mine):
        tokens = tokenize(d["text"]); my_len.append(len(tokens))
        for word, tf in Counter(tokens).items():
            my_index[word][i] = tf
    my_avg = sum(my_len) / len(mine)
    def my_bm25(query, k1=1.2, b=0.75):
        scores = np.zeros(len(mine))
        for w in tokenize(query):
            p = my_index.get(w, {})
            w_idf = math.log(1 + (len(mine) - len(p) + 0.5) / (len(p) + 0.5))
            for i, tf in p.items():
                scores[i] += w_idf * tf * (k1 + 1) / (tf + k1 * (1 - b + b * my_len[i] / my_avg))
        return scores
    my_vecs = encode([d["text"] for d in mine], "passage")
    rankings_record = []
    lines = [f"# Поиск по своей библиотеке: {MY_TOPIC}", "",
             f"[источник] OpenAlex, запрос: {request}, дата: {date.today().isoformat()}, работ: {len(mine)}", ""]
    for q in MY_QUERIES:
        sb, sv = my_bm25(q), my_vecs @ encode([q], "query")[0]
        lines += [f"## {q}", "", "| место | BM25 | вектор |", "|---|---|---|"]
        rb = [int(i) for i in np.argsort(-sb)[:3] if sb[i] > 0]; rv = [int(i) for i in np.argsort(-sv)[:3]]
        rankings_record.append({"query": q, "bm25": [{"id": mine[i]["id"], "score": float(sb[i])} for i in rb],
                                "vector": [{"id": mine[i]["id"], "score": float(sv[i])} for i in rv]})
        for r in range(3):
            left = f"{mine[rb[r]]['title'][:60]} ({mine[rb[r]]['id']})" if r < len(rb) else "—"
            lines.append(f"| {r + 1} | {left} | {mine[rv[r]]['title'][:60]} ({mine[rv[r]]['id']}) |")
        lines += ["", "[написано] Что подошло и чем выдачи отличаются: …", ""]
    (PROJECT / "reports/my_search_report.md").write_text("\n".join(lines), encoding="utf-8")
    (PROJECT / "library_search/rankings.json").write_text(json.dumps(rankings_record, ensure_ascii=False, indent=2), encoding="utf-8")
    print("\n".join(lines[:16])); print("…\nзаписан файл my_search_report.md; допишите строки [написано]")
else:
    print("Заполните MY_TOPIC и MY_QUERIES и выполните ячейку.")

NameError: name 'PROJECT' is not defined